# 02_build_final_sample_focus

**Objective:** Build the focal analytical sample of patent-holding startups that filed at least one patent before their first equity round, deriving firm-level features and flagging excluded firms.

**Inputs:** `1_startups_with_patents.csv`; `startup_patent_matches.csv`.

**Outputs:** `2_final_sample_focus.csv`.

## Imports

In [ ]:
# Imports
import pandas as pd
from pathlib import Path

## Configuration: paths, winsorise columns, and final column order

In [ ]:
# Configuration: paths, winsorise columns, and final column order
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
SRC          = PROC / '1_startups_with_patents.csv'
MATCHES_PATH = PROC / 'startup_patent_matches.csv'
OUT          = PROC / '2_final_sample_focus.csv'

In [ ]:
# Final column order grouped by analytical role
COL_ORDER = [
    'Organization Name', 'Organization Name URL', 'country',
    'excluded_startup',
    'exit_binary', 'exit_type', 'years_to_exit',
    'total_funding_usd', 'log_total_funding_usd',
    'bucket_late',
    'stage_pre_seed', 'stage_seed', 'stage_angel', 'stage_series_a',
    'stage_series_b', 'stage_series_c_plus', 'stage_private_equity', 'stage_post_ipo',
    'founding_year', 'tech_field_group', 'age_at_first_funding',
    'n_patents', 'n_patents_with_novelty',
    'novelty_mean', 'novelty_max', 'competitive_density', 'competitive_density_max',
    'bwd_cits_mean',    'bwd_cits_max',
    'npl_cits_mean',    'npl_cits_max',
    'originality_mean', 'originality_max',
    'radicalness_mean', 'radicalness_max',
    'patent_scope_mean', 'patent_scope_max',
    'claims_mean',       'claims_max',
    'family_size_mean',  'family_size_max',
]

## Main pipeline: sample selection, feature derivation, winsorise, exclusion flag, reorder, write

In [ ]:
# Main pipeline: select sample, derive features, winsorise, flag exclusions, save
def main():
    df = pd.read_csv(SRC)

    sample = (
        df['first_equity_round_date'].notna()
        & (df['n_patents'] > 0)
        & (df['n_patents_with_novelty'] > 0)
    )
    df = df[sample].copy()
    print(f'Sample (strictly measured): {len(df):,} rows')

    df['competitive_density']     = 1.0 - df['novelty_top10_mean']
    df['competitive_density_max'] = 1.0 - df['novelty_top10_max']

    df['first_equity_round_date'] = pd.to_datetime(df['first_equity_round_date'], errors='coerce')

    matches = pd.read_csv(MATCHES_PATH, usecols=['Organization Name', 'earliest_priority'])
    matches['earliest_priority'] = pd.to_datetime(matches['earliest_priority'], errors='coerce')

    cutoff = df[['Organization Name', 'first_equity_round_date']]
    mm = matches.merge(cutoff, on='Organization Name', how='inner')
    mm = mm[mm['earliest_priority'] < mm['first_equity_round_date']]
    mm['priority_year'] = mm['earliest_priority'].dt.year

    min_pyr = (mm.groupby('Organization Name')['priority_year']
                 .min()
                 .rename('_min_pre_eq_priority_year'))
    df = df.merge(min_pyr, on='Organization Name', how='left')

    rule_a = (df['years_to_exit'] < 0).fillna(False)
    rule_b = (df['age_at_first_funding'] < -0.5).fillna(False)
    rule_c = ((df['n_patents'] > 50)
              & ((df['founding_year'] - df['_min_pre_eq_priority_year']) >= 5)).fillna(False)
    df['excluded_startup'] = (rule_a | rule_b | rule_c).astype(int)
    df = df.drop(columns=['_min_pre_eq_priority_year'])

    print(f'\nexcluded_startup flag triggered: {int(df["excluded_startup"].sum())} rows')
    print(f'  rule a -- years_to_exit < 0                 : {int(rule_a.sum())}')
    print(f'  rule b -- age_at_first_funding < -0.5       : {int(rule_b.sum())}')
    print(f'  rule c -- >50 patents AND >=5y pre-founding : {int(rule_c.sum())}')
    print(f'  (rules overlap; total flagged is the union)')

    missing = [c for c in COL_ORDER if c not in df.columns]
    if missing:
        raise RuntimeError(f'Missing expected columns: {missing}')
    df = df[COL_ORDER]

    df.to_csv(OUT, index=False)
    print(f'\nWrote: {OUT}  ({df.shape[0]:,} rows x {df.shape[1]} cols)')

## Run when executed as a script

In [ ]:
# Entry point
if __name__ == '__main__':
    main()